In [ ]:
from google.colab import drive

# Mount Google Drive (optional)
drive.mount('/content/drive')

# Clone your GitHub repository
!git clone https://github.com/beingAnujChaudhary/LLM-Zoomcamp.git

# Move into repository
%cd /content/LLM-Zoomcamp

# Move into Chapter 1 folder
%cd 02-vector-search

# 📘 The Vector Search 

---

Search is fundamentally about connecting a user's intent to the right data. Historically, this meant looking for exact word matches. Today, modern applications use Artificial Intelligence to understand the meaning behind a query. This masterclass bridges the gap between traditional search methodologies and modern, AI-driven Vector Search.

## Part 1: The Evolution of Information Retrieval 
Information Retrieval (IR) is the science of searching for documents, information within documents, and metadata about documents. 
*   **Boolean Era (1960s):** Exact matching using AND/OR/NOT operators.
*   **Probabilistic/Lexical Era (1970s-2000s):** TF-IDF and BM25. Matching based on word frequency and statistical rarity.
*   **Semantic/Dense Era (2010s-Present):** Word2Vec, Transformers, and Vector Search. Matching based on geometric proximity in high-dimensional space.

---


## Part 2: Keyword Search & The Semantic Gap 

### 4. TF-IDF (Term Frequency-Inverse Document Frequency)
TF-IDF penalizes words that appear too frequently across all documents (like "the" or "is") and boosts words that are frequent in a specific document but rare globally.
*   **Formula:** $TF\text{-}IDF(t, d) = TF(t, d) \times \log\left(\frac{N}{df(t)}\right)$
  *   $TF$: Frequency of term $t$ in document $d$.
  *   $N$: Total number of documents.
  *   $df(t)$: Number of documents containing term $t$.

### 5. BM25 (Best Matching 25)
BM25 is the industry-standard probabilistic refinement of TF-IDF used in Elasticsearch and Solr. It introduces **term frequency saturation** and **document length normalization**.
*   **Formula:** $Score(D,Q) = \sum_{i=1}^{n} IDF(q_i) \times \frac{f(q_i,D) \cdot (k_1 + 1)}{f(q_i,D) + k_1 \cdot (1 - b + b \cdot \frac{|D|}{avgdl})}$
  *   $k_1$: Saturation parameter (usually 1.2 - 2.0). Prevents a term appearing 100 times from scoring 100x higher than appearing 10 times.
  *   $b$: Length normalization (usually 0.75). Penalizes longer documents.

### 6-7. Why Keyword Search Fails (The Semantic Gap)
Keyword search suffers from the **Vocabulary Mismatch Problem**. 
*   **Query:** *"Can I still join the course after the start date?"*
*   **Document:** *"Is it possible to enroll late?"*
*   **Lexical Overlap:** ~10% (only "the" or "I"). 
*   **Semantic Overlap:** 100%.
Keyword search fails here because it lacks context, synonym handling, and paraphrase understanding.

---


## Part 3: Semantic Search & Embeddings 

### 8. Semantic Search
Semantic search overcomes the Vocabulary Mismatch Problem by matching ideas and meaning rather than exact characters. If a user searches for "feline", semantic search knows to retrieve documents containing the word "cat", even if "feline" is completely absent from the text.

Instead of matching discrete tokens, semantic search maps text into a continuous, dense mathematical space where **semantic meaning translates to geometric proximity**.

### 9. Embeddings (Word vs. Sentence)

An **Embedding** is a mathematical representation of text as a dense vector (an array of numbers or $N$ floating-point numbers) in a continuous, multi-dimensional space.

*   **Static Word Embeddings (Word2Vec, GloVe):** Map individual words to vectors. *Flaw:* The word "bank" gets the same vector whether you mean a river bank or a financial bank (Polysemy problem).
*   **Contextual Sentence Embeddings (BERT, Sentence-Transformers):** Use the **Transformer architecture** and **self-attention** to encode entire sentences. The vector for "judge" changes dynamically based on surrounding words (e.g., "court" vs "evaluate LLM").

---



## Part 4: Mathematical Foundations 

### 10-11. Linear Algebra & Vector Space

A vector is an ordered array of numbers (e.g., `[0.12, -0.34, 0.56]`).

*   **Dimensionality:** The number of components in the array (e.g., standard models use 384, 768, or 1536 dimensions).
*   **Magnitude (Norm):** The physical length or magnitude of the vector.
*   **Direction:** The orientation in the multi-dimensional space.

When we process thousands of documents, each one becomes a single point in a high-dimensional **vector space**.

* **Semantic Proximity:** Because the neural network was trained on human language, vectors with similar meanings naturally land near each other geometrically.
* **Dense Vectors:** Unlike one-hot encoding, most values in these arrays are non-zero.

### 12. Similarity Metrics & The Normalization Trick
To measure how "close" two vectors are, we need a distance metric.

**1. Cosine Similarity:** Measures the *angle* ($\theta$) between vectors, ignoring magnitude.
$$ \cos(\theta) = \frac{A \cdot B}{||A||_2 \times ||B||_2} $$
*Range: [-1, 1]. (1 = identical, 0 = orthogonal, -1 = opposite).*

**2. Dot Product (Inner Product):** The algebraic sum of corresponding components.
$$ A \cdot B = \sum_{i=1}^{n} A_i B_i $$

**3. Euclidean Distance (L2):** Straight-line physical distance. Sensitive to magnitude.

> 💡 **CRUCIAL TECHNICALITY: The Normalization Trick**
> Calculating the denominator in Cosine Similarity ($||A||_2 \times ||B||_2$) is computationally expensive. If we **L2-normalize** our vectors during indexing (scaling them so their length $||A||_2 = 1$), the denominator becomes $1 \times 1 = 1$. 
> **Therefore, for normalized vectors, Cosine Similarity is mathematically identical to the Dot Product.** 
> Vector databases (like FAISS) use Dot Product indexes (`IndexFlatIP`) under the hood because it is significantly faster to compute.

---


## Part 5: The Vector Pipeline & ANN Algorithms 

### 13. The Two-Phase Pipeline
1.  **Offline (Indexing):** Pass all documents through an Embedding Model to convert them into vectors and store them in a Vector Database.

Text $\rightarrow$ Chunking $\rightarrow$ Embedding Model $\rightarrow$ Vector Arrays $\rightarrow$ Vector Index.


2.  **Online (Querying):** When a user asks a question, pass the query through the exact same model. The system searches the database for document vectors closest to the query vector

Query $\rightarrow$ Embedding Model $\rightarrow$ Query Vector $\rightarrow$ ANN Search $\rightarrow$ Top-K Results.

### 14. Approximate Nearest Neighbors (ANN)
Brute-force search (calculating distance against every vector) is $O(N)$. For 10 million documents, this is too slow. Production systems use ANN to achieve $O(\log N)$ or $O(\sqrt{N})$ speed.
*   **HNSW (Hierarchical Navigable Small World):** Builds a multi-layered graph for rapid traversal. Top layers are sparse (allowing massive "jumps" across the space), bottom layers are dense (allowing precise local search). *Industry standard for high-recall, low-latency.*
*   **IVF (Inverted File Index):** Uses K-Means clustering to divide the space into "Voronoi cells". Search only occurs in the closest clusters.
*   **PQ (Product Quantization):** Compresses vectors by breaking them into sub-vectors and clustering them, drastically reducing RAM usage.

---


## Part 6: Vector Databases 

* **Vector Extensions**: Tools like `pgvector` or `sqlite-vec` add vector math capabilities to existing relational databases, ensuring ACID transactions.

* **Native Vector DBs**: Platforms like `Pinecone, Milvus, Qdrant, and ChromaDB` are built from the ground up for massive-scale vector math, advanced filtering, and clustering.

| Tool | Architecture | Pros | Cons |
| :--- | :--- | :--- | :--- |
| **Minsearch** | In-Memory (Python) | Simple, zero-dependency, great for prototyping. | $O(N)$ brute force, lost on restart. |
| **SQLite-vec** | Relational Extension | ACID compliant, keeps metadata and vectors together. | Not optimized for massive scale. |
| **PGVector** | PostgreSQL Extension | Enterprise-ready, SQL ecosystem, highly scalable. | Requires Postgres expertise, heavy setup. |
| **Pinecone/Qdrant** | Native Vector DB | Built from scratch for vectors, managed, distributed. | Separate infrastructure stack, vendor lock-in. |

---


## Part 7: RAG Integration & Production Best Practices 

Because Vector Search is bad at exact term matching (like specific IDs), modern Retrieval-Augmented Generation (RAG) systems use Hybrid Search.
This runs both a Keyword Search and a Vector Search simultaneously. Because BM25 scores and Cosine scores are on different mathematical scales, they are merged using Reciprocal Rank Fusion (RRF), which scores based on rankings rather than raw distances.

### 1. Document Chunking
Embedding models perform best on small, coherent blocks of text.
*   **Recursive Character Splitting:** Splits by paragraphs $\rightarrow$ sentences $\rightarrow$ words to preserve semantic units.
*   **Overlap:** Crucial! You must overlap chunks (e.g., 500 tokens with 50 tokens overlap) so context at the boundaries isn't severed.

### 2. Hybrid Search & Reciprocal Rank Fusion (RRF)
Vector search fails at exact keyword matching (e.g., searching for an error code like `ERR-404`). **Hybrid Search** runs BM25 and Vector search simultaneously.
Because BM25 scores and Cosine scores are on different scales, we cannot simply add them. We use **RRF**, which ignores raw scores and only looks at rank:
$$ RRF\_score(d) = \sum_{r \in R} \frac{1}{k + rank_r(d)} $$
*(where $k$ is a constant, usually 60, to prevent top ranks from dominating).*

### 3. Quantization
Storing millions of 384-dimensional Float32 vectors requires massive RAM. **Scalar Quantization** converts 32-bit floats to 8-bit integers, reducing memory footprint by 4x with only a ~1% drop in accuracy.

### 4. Evaluation Metrics
*   **Precision@K:** Out of the top K retrieved, how many are actually relevant?
*   **Recall@K:** Out of all relevant documents in the DB, how many were found in the top K?
*   **NDCG (Normalized Discounted Cumulative Gain):** Measures ranking quality (penalizes relevant documents appearing lower in the list).

### Production Best Practices
* **Start Simple**: Never start with vector search. Start with basic text search and upgrade to hybrid/vector once the operational complexity is justified.

* **Chunking**: LLMs have context limits. Overlap chunks (e.g., 500 tokens with 50 tokens overlap) so context isn't lost at the boundaries.

* **Quantization**: Compress high-dimensional vectors (e.g., 32-bit floats down to 8-bit integers) to reduce memory footprints by 4x to 32x with only a 1-2% drop in accuracy.
---


## Part 8: Interview Questions, Exercises & Cheat Sheet 

### 🎤 Top 5 Interview Questions
1.  **Q: Why do production vector databases prefer Dot Product over Cosine Similarity?**
    *   *A:* Because vectors are L2-normalized during indexing. For normalized vectors, Dot Product is mathematically identical to Cosine Similarity but requires significantly fewer CPU/GPU cycles to compute.
2.  **Q: Explain how HNSW works.**
    *   *A:* HNSW builds a multi-layered graph. The top layers are sparse, allowing the search algorithm to quickly jump across distant regions of the vector space. As it descends to the bottom layers, the graph becomes dense, allowing for precise, localized neighbor traversal.
3.  **Q: What is the Curse of Dimensionality in the context of vector search?**
    *   *A:* As dimensions increase, the volume of the space increases exponentially, making the data sparse. In high dimensions, the difference in Euclidean distance between the "nearest" and "farthest" points approaches zero, making distance metrics less meaningful. This is why Cosine Similarity (angle) is preferred over Euclidean (distance) for high-dimensional embeddings.
4.  **Q: Why use Hybrid Search instead of just Vector Search?**
    *   *A:* Vector search is "black box" and struggles with exact matches (like IDs, SKUs, or specific error codes). Keyword search (BM25) excels at exact matches but fails on synonyms. Hybrid search combines both to cover all retrieval bases.
5.  **Q: What is Reciprocal Rank Fusion (RRF)?**
    *   *A:* RRF is an algorithm used to merge results from multiple search retrievers (like BM25 and Vector) that have incomparable score scales. It calculates a new score based purely on the document's rank in each list: $1 / (k + rank)$.

### 🛠️ Practical Exercises
1.  **Exercise 1:** Modify the  code below to use `IndexIVFFlat` in FAISS instead of `IndexFlatIP`. Measure the difference in search time for 10,000 random vectors.
2.  **Exercise 2:** Implement a basic RRF algorithm in Python that takes two lists of document IDs (one from BM25, one from Vector) and merges them.
3.  **Exercise 3:** Take a dataset of 100 FAQ questions. Embed them, and use UMAP to reduce the dimensions to 2D. Plot them using `matplotlib` to visually verify that similar questions cluster together.

### 📝 Quick Cheat Sheet
| Concept | Definition / Formula |
| :--- | :--- |
| **Embedding** | Dense array of floats representing semantic meaning. |
| **Cosine Similarity** | $\frac{A \cdot B}{||A|| ||B||}$. Measures angle. Range: [-1, 1]. |
| **Dot Product** | $A \cdot B$. Equals Cosine Sim if vectors are L2-normalized. |
| **HNSW** | Graph-based ANN. High memory, extremely fast, high recall. |
| **IVF** | Cluster-based ANN. Lower memory, requires training step. |
| **RRF** | $\sum \frac{1}{k + rank}$. Merges hybrid search results. |
| **Chunking** | Splitting text. Always use overlap (e.g., 10-20%) to preserve context. |

---


## Part 9: Code Implementation

### Cell 1: Install Dependencies
```python
# Install required libraries
# sentence-transformers: For generating embeddings
# faiss-cpu: Facebook's library for highly efficient similarity search (ANN)
# scikit-learn: For L2 normalization and TF-IDF comparison
!pip install -q sentence-transformers faiss-cpu scikit-learn numpy
```

### Cell 2: Setup Data and Generate Embeddings
```python
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Load a lightweight, fast embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Define a sample dataset (FAQ style)
documents = [
    "Can I still join the course after the start date?",
    "Is it possible to enroll late?",
    "How do I run Docker on Windows?",
    "What is the refund policy for the bootcamp?",
    "pandas dataframe tutorial for beginners"
]

# 3. Generate raw embeddings
raw_embeddings = model.encode(documents, convert_to_numpy=True)
print(f"Raw Embeddings Shape: {raw_embeddings.shape}") # (5 documents, 384 dimensions)

# 4. L2 NORMALIZATION (The Crucial Technical Step)
# We scale the vectors so their length (magnitude) is exactly 1.
normalized_embeddings = normalize(raw_embeddings, norm='l2')

# Verify they are normalized (the L2 norm of each row should be 1.0)
print(f"L2 Norms after normalization: {np.linalg.norm(normalized_embeddings, axis=1)}")
```

### Cell 3: The Semantic Gap (TF-IDF vs Vector Search)
```python
# Let's prove why Keyword search fails and Vector search succeeds.
query = "Can I join late?"

# --- KEYWORD SEARCH (TF-IDF) ---
vectorizer = TfidfVectorizer()
doc_tfidf = vectorizer.fit_transform(documents)
query_tfidf = vectorizer.transform([query])
tfidf_scores = cosine_similarity(query_tfidf, doc_tfidf)[0]

# --- VECTOR SEARCH (Semantic) ---
query_raw = model.encode([query], convert_to_numpy=True)
query_norm = normalize(query_raw, norm='l2')
# Using Dot Product (which equals Cosine Sim for normalized vectors)
vector_scores = np.dot(normalized_embeddings, query_norm.T).flatten()

print("\n--- The Semantic Gap: 'Can I join late?' ---")
print("Document".ljust(50), "| TF-IDF (Keyword)", "| Vector (Semantic)")
print("-" * 85)
for doc, tfidf, vec in zip(documents, tfidf_scores, vector_scores):
    print(f"{doc[:48].ljust(50)} | {tfidf:.4f}           | {vec:.4f}")
    
# Notice: TF-IDF gives 0.0 for "enroll late" because the words don't match.
# Vector search gives a high score because the MEANING matches!
```

### Cell 4: Mathematical Proof (Dot Product == Cosine Similarity)
```python
# Let's mathematically prove the Normalization Trick.

# Method A: Scikit-Learn Cosine Similarity (Calculates angles, slower)
cos_scores = cosine_similarity(query_norm, normalized_embeddings)[0]

# Method B: Dot Product (Algebraic sum, much faster)
dot_scores = np.dot(normalized_embeddings, query_norm.T).flatten()

print("\n--- Mathematical Proof ---")
print("Are Cosine Similarity and Dot Product identical?", np.allclose(cos_scores, dot_scores))
```

### Cell 5: Production-Ready Search with FAISS (ANN)
```python
# In production, we don't use brute force (O(N)). We use ANN indexes.
# FAISS (Facebook AI Similarity Search) is the industry standard.

dimension = normalized_embeddings.shape[1] # 384

# IndexFlatIP = Flat (Brute force for now, but easily swappable to HNSW) 
# IP = Inner Product (Dot Product). 
index = faiss.IndexFlatIP(dimension)

# Add vectors to the index
index.add(normalized_embeddings)

# Perform a search for the top 3 closest documents
k = 3
distances, indices = index.search(query_norm, k)

print(f"\n--- FAISS Top {k} Results (Using Inner Product) ---")
for i in range(k):
    doc_idx = indices[0][i]
    score = distances[0][i]
    print(f"Score: {score:.4f} | Document: {documents[doc_idx]}")

# Note: To switch to a true Approximate Nearest Neighbor (HNSW) index for millions of docs, 
# you would simply change the index initialization to:
# index = faiss.IndexHNSWFlat(dimension, 32) # 32 is the number of connections per node (M)
```